# Processing

## Setup & Data Ingestion


In [ ]:
import warnings, re, sys, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates  # Required for time-series formatting
import seaborn as sns
from scipy.sparse import lil_matrix, csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from numpy.linalg import norm


sns.set_style("whitegrid")
DARK = "#0E1117"
plt.rcParams.update({
    "figure.dpi": 150, "font.family": "sans-serif",
    "axes.titleweight": "bold", "axes.labelweight": "bold",
    "figure.facecolor": DARK, "axes.facecolor": DARK,
    "text.color": "#E0E0E0", "axes.labelcolor": "#E0E0E0",
    "xtick.color": "#AAAAAA", "ytick.color": "#AAAAAA",
    "grid.color": "#2A2D35", "grid.alpha": 0.5,
})
ACCENT = ["#FF6B6B", "#4ECDC4", "#FFE66D", "#A06CD5", "#FF9F43",
          "#54A0FF", "#5CD85C", "#FD7272", "#C56CF0", "#17C0EB"]

In [ ]:
# Data Ingestion & Cleaning

# Define tags that would be excluded
NOISE_TAGS = frozenset({
    "malevocals", "femalevocals", "maleandfemalevocals",
    "instrumental", "vocals", "vocal", "conceptalbum", "concept",
    "album", "acoustic", "electric", "production", "lo-fiproduction",
    "liveperformance", "live", "liverecording", "studio", "studioalbum",
    "remaster", "remastered", "reissue", "compilation", "soundtrack",
    "ost", "debut", "debutalbum", "sideproject", "collaboration",
    "ep", "single", "mixtape", "bonus", "bonustracks", "deluxe",
    "explicit", "clean", "mono", "stereo",
    "avant-garde", "choral", "improvisation", "monologue",
    "opera", "orchestral", "rockopera", "suite", "carol",
    "ensemble", "vocalgroup", "wallofsound", "androgynousvocals",
    "lo-fi", "symphony", "jingle", "chambermusic", "generativemusic",
    "sonata", "acappella", "...",

    "melodic", "energetic"
})

def load_data(path: str) -> pd.DataFrame:
    """
    Load and clean the dataset:
    1. Parse comma-separated descriptors into a list of tags.
    2. Filter out noise tags and empty lists.
    3. Parse release dates to datetime objects
    """
    df = pd.read_csv(path, encoding="latin-1")

    def clean_descriptors(raw):
        if pd.isna(raw):
            return []
        # Split, strip, lower, and filter
        tags = [t.strip().lower().replace(" ", "") for t in str(raw).split(",")]
        return [t for t in tags if t and len(t) > 1 and t not in NOISE_TAGS]

    df["tags"] = df["descriptors"].apply(clean_descriptors)

    # filter out rows with no valid tags
    df = df[df["tags"].apply(len) > 0].reset_index(drop=True)

    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

    # Drop rows where date parsing failed to ensure valid timeline
    df = df.dropna(subset=["release_date"]).reset_index(drop=True)

    # Normalize to the first day of the month for plotting

    df["date_obj"] = df["release_date"].dt.to_period("M").dt.to_timestamp()

    # Maintaining 'year' column for compatibility of old codes
    df["year"] = df["date_obj"].dt.year.astype(int)

    df = df[['position', 'release_name', 'artist_name', 'primary_genres', 'secondary_genres', 'avg_rating', 'rating_count', 'review_count', 'year', 'date_obj', 'tags',]]

    return df

In [ ]:
DATA_PATH = "rym_clean1.csv"
df = load_data(DATA_PATH)

print(f"Shape: {df.shape}")
display(df.head())

## Define a Positional Decay Function for TFIDF  (Graph Explanation)

In [ ]:
def positional_decay_weights(n: int, alpha: float = 0.50) -> np.ndarray:
    """
    Compute exponential positional decay vector for a tag list of length n.

    n     : number of tags in the album's descriptor list

    alpha : floor weight for the last tag (default 0.50 since the descriptor was alread top (10 or so) tags from RYM)

    Returns an np.ndarray with w[0]=1.0 and w[n-1]≈alpha
    """

    if n <= 1:
        return np.ones(n, dtype=np.float64)
    # λ = -ln(α) / (n - 1)
    lam = -np.log(alpha) / (n - 1)
    return np.exp(-lam * np.arange(n))


def plot_decay_comparison(save_path="fig_decay_curves.png"):
    """
    Compare exponential vs harmonic vs square-root decay curves.
    """
    n = 10
    positions = np.arange(n)


    w_exp = positional_decay_weights(n, alpha=0.50)
    w_harm = 1.0 / (positions + 1)
    w_sqrt = 1.0 / np.sqrt(positions + 1)


    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(DARK)

    ax.plot(positions, w_exp, "o-", color=ACCENT[0],
            label="Exponential (Proposed)  w(i)=exp(-λi)")
    ax.plot(positions, w_sqrt, "s--", color=ACCENT[5],
            label="Square-root  w(i)=1/√(i+1)")
    ax.plot(positions, w_harm, "D:", color=ACCENT[2],
            label="Harmonic  w(i)=1/(i+1)")



    ax.set_xticks(positions)
    ax.set_xticklabels([f"Tag {i}" for i in range(n)])
    ax.set_xlabel("Descriptor Order")
    ax.set_ylabel("Weight")
    ax.set_title(f"Positional Decay Function Comparison (n={n}, α=0.5)",
                 fontsize=13, pad=12, color="white")
    ax.set_ylim(-0.05, 1.15)

    leg = ax.legend(fontsize=10, framealpha=0.2)
    for t in leg.get_texts():
        t.set_color("white")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"  → Saved comparison plot to: {save_path}")


# Generate the comparison plot
plot_decay_comparison()

## Weighted TF-IDF

In [ ]:
def build_weighted_tfidf(df: pd.DataFrame):
    """
    Construct a positionally-weighted TF-IDF sparse matrix.

    Returns
    -------
    M_norm  : csr_matrix (n_albums × n_vocab), L2-normalized rows
    vocab   : sorted list of unique tags
    tag2col : dict mapping tag string → column index
    idf     : np.ndarray (n_vocab,) of IDF values
    """

    tag_sets = df["tags"].apply(set)
    vocab_counter = {}
    for ts in tag_sets:
        for t in ts:
            vocab_counter[t] = vocab_counter.get(t, 0) + 1

    vocab = sorted(vocab_counter.keys())
    tag2col = {t: i for i, t in enumerate(vocab)}
    n_albums = len(df)
    n_vocab = len(vocab)

    # IDF: smooth variant  log( (1 + N) / (1 + df_t) ) + 1
    df_arr = np.array([vocab_counter[t] for t in vocab], dtype=np.float64)
    idf = np.log((1.0 + n_albums) / (1.0 + df_arr)) + 1.0


    # Fill the sparse matrix with positional weights × IDF
    # Use LIL for efficient incremental assignment
    M = lil_matrix((n_albums, n_vocab), dtype=np.float64)

    for row_idx, tags in enumerate(df["tags"]):
        n = len(tags)
        # Reuse the function from Step 1
        weights = positional_decay_weights(n, alpha=0.50)
        for pos, tag in enumerate(tags):
            if tag in tag2col:
                col = tag2col[tag]
                # Weighted TF-IDF: positional_weight * idf
                M[row_idx, col] = weights[pos] * idf[col]

    M_csr = M.tocsr()

    # L2-normalize each album (row)
    # Essential for cosine similarity
    M_norm = normalize(M_csr, norm="l2", axis=1)

    sparsity = 1.0 - M_csr.nnz / (n_albums * n_vocab)


    print(f"  Vocabulary : {n_vocab} tags")
    print(f"  IDF range  : [{idf.min():.3f}, {idf.max():.3f}]")
    print(f"  Albums     : {n_albums}")
    print(f"  Sparsity   : {sparsity:.2%}")
    print(f"  NNZ entries: {M_csr.nnz:,}")

    return M_norm, vocab, tag2col, idf

# Execution
M, vocab, tag2col, idf = build_weighted_tfidf(df)

# (NOT IMPORTANT) Visualization

## (NOT IMPORTANT) Evolution of Band on Specific Descriptor

In [ ]:
def plot_descriptor_evolution(artist_name: str, target_descriptor: str = None,
                              M = None, df = None,
                              tag2col = None, vocab = None,
                              save_path: str = "./"):

    """
    Visualize the evolution of descriptor weights for an artist over time.
    Uses month-level granularity for the x-axis.
    """

    # Handle Globals if not provided (Safety fallback for notebook convenience)
    if M is None: M = globals().get('M')
    if df is None: df = globals().get('df')
    if tag2col is None: tag2col = globals().get('tag2col')
    if vocab is None: vocab = globals().get('vocab')

    if M is None or df is None:
        print("Error: Matrix M or DataFrame df not provided/found.")
        return

    # Filter and Sort by new date object
    mask = df["artist_name"].str.contains(artist_name, case=False, na=False)
    # Refactored: Sort by 'date_obj' instead of 'year' for accurate intra-year ordering
    artist_df = df[mask].sort_values("date_obj")

    if artist_df.empty:
        print(f"Artist '{artist_name}' not found.")
        return

    indices = artist_df.index.values
    dates = artist_df["date_obj"].values  # X-axis data: Datetime objects
    titles = artist_df["release_name"].values

    # Isolate Rows from TF-IDF Matrix
    sub_M = M[indices].toarray()  # (n_albums, n_vocab)

    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor(DARK)

    # 3. Extraction Logic
    if target_descriptor:
        # Case A: Specific Descriptor
        clean_tag = target_descriptor.strip().lower().replace(" ", "")
        if clean_tag not in tag2col:
            print(f"Descriptor '{target_descriptor}' not in vocabulary.")
            return

        col_idx = tag2col[clean_tag]
        weights = sub_M[:, col_idx]

        # Refactored: Plot against actual dates
        ax.plot(dates, weights, "o-", color=ACCENT[0], linewidth=2.5,
                label=clean_tag)
        ax.fill_between(dates, weights, alpha=0.1, color=ACCENT[0])
        title_suffix = f"Evolution of '{clean_tag}'"

    else:
        # Case B: Top 3 High-Variance Descriptors
        variances = np.var(sub_M, axis=0)
        top_3_idx = np.argsort(variances)[-3:][::-1]

        for i, idx in enumerate(top_3_idx):
            tag = vocab[idx]
            weights = sub_M[:, idx]
            # Refactored: Plot against actual dates
            ax.plot(dates, weights, "o-", color=ACCENT[i % len(ACCENT)],
                    linewidth=2, label=tag, alpha=0.9)

        title_suffix = "Top 3 Volatile Descriptors"

    # Visualization Styling with mdates
    ax.set_ylabel("Weighted TF-IDF Score", fontsize=10)
    ax.set_title(f"{artist_df.iloc[0]['artist_name']} — {title_suffix}",
                 fontsize=14, color="white", pad=15)
    ax.grid(True, alpha=0.15)

    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

    # Rotate labels to prevent overlap and ensure readability
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9, color="#AAAAAA")

    leg = ax.legend(fontsize=10, framealpha=0.2)
    for t in leg.get_texts(): t.set_color("white")

    plt.tight_layout()

    filename = f"fig_evolution_{artist_name.replace(' ', '')}.png"
    full_path = f"{filename}"
    plt.savefig(full_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()

### Excution

In [ ]:
# Execution
print("Specific Descriptor")
plot_descriptor_evolution("Metallica", target_descriptor="heavy")

print("\n Most Volatile Descriptor")
plot_descriptor_evolution("Pink Floyd")

# Processing

## Latent Semantic Analysis using Truncated SVD

In [ ]:
def fit_svd_with_elbow(M: csr_matrix, k_candidates=None, plot_path=None):
    """
    Fit Truncated SVD and select optimal components via elbow analysis.

    Returns
    -------
    Z        : np.ndarray (n_albums, k_opt) — dense latent representations
    svd      : fitted TruncatedSVD object
    k_opt    : selected number of components
    """

    if k_candidates is None:
        k_candidates = list(range(5, 105, 5))

    # Ensure we don't exceed matrix dimensions
    max_k = min(max(k_candidates), min(M.shape) - 1)

    print(f"\n  ── SVD Elbow Analysis ──")
    print(f"  Sweeping k ∈ [{min(k_candidates)}, {max_k}] ...")

    # 1. Fit full model up to max_k
    svd_full = TruncatedSVD(n_components=max_k, random_state=42, algorithm="randomized")
    svd_full.fit(M)

    # Cumulative explained variance
    cum_var = np.cumsum(svd_full.explained_variance_ratio_)

    # Filter candidates to those actually fitted
    valid_candidates = [k for k in k_candidates if k <= max_k]
    candidate_vars = [cum_var[k-1] for k in valid_candidates]

    # Elbow detection
    # We use the second derivative of the explained variance curve.
    # Since the curve is discrete, we approximate with finite differences.

    ks = np.array(valid_candidates)
    vs = np.array(candidate_vars)

    if len(vs) >= 3:
        # First derivative (marginal gain)
        d1 = np.diff(vs)
        # Second derivative (change in marginal gain)
        d2 = np.diff(d1)
        # The elbow is typically where the gain drops most significantly (min 2nd deriv)
        # +1 because diff reduces length by 1, and we want the index in `ks`
        elbow_idx = np.argmin(d2) + 1
        k_opt = ks[elbow_idx]

        # Enforce minimum k
        k_opt = max(k_opt, 30)
    else:
        k_opt = 50  # Fallback

    # Capture k_opt
    k_opt = min(k_opt, max_k)
    var_at_k = cum_var[k_opt - 1]

    print(f"  Elbow chosen at k = {k_opt}")
    print(f"  Explained variance at k={k_opt}: {var_at_k:.2%}")
    print(f"  Variance at max k={max_k}: {cum_var[-1]:.2%}")

    # Visualization
    if plot_path:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        fig.patch.set_facecolor(DARK)

        # Left: Cumulative Variance
        ax1.plot(ks, vs, "o-", color=ACCENT[1], linewidth=2, markersize=5)
        ax1.axvline(k_opt, color=ACCENT[0], linestyle="--", linewidth=1.5,
                    label=f"Chosen: k={k_opt} ({var_at_k:.1%})")
        ax1.axhline(var_at_k, color=ACCENT[0], linestyle=":", alpha=0.4)
        ax1.set_xlabel("SVD Components (k)", fontsize=11)
        ax1.set_ylabel("Cumulative Explained Variance", fontsize=11)
        ax1.set_title("Cumulative Variance", fontsize=12, pad=10, color="white")
        ax1.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
        leg = ax1.legend(fontsize=9, framealpha=0.3)
        for t in leg.get_texts(): t.set_color("white")

        # Right: Marginal Gain
        # Use the raw explained variance ratio for the bar chart blocks
        # We approximate the gain of a block of 5 components
        marginal_gains = np.diff([0] + list(vs))
        bars = ax2.bar(ks, marginal_gains, width=3.5, color=ACCENT[5], alpha=0.75, edgecolor="#333")

        # Highlight elbow
        try:
            idx = list(ks).index(k_opt)
            bars[idx].set_color(ACCENT[0])
            bars[idx].set_alpha(1.0)
        except ValueError:
            pass

        ax2.set_xlabel("SVD Components (k)", fontsize=11)
        ax2.set_ylabel("Marginal Variance Gain", fontsize=11)
        ax2.set_title("Marginal Gain", fontsize=12, pad=10, color="white")
        ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

        plt.tight_layout()
        plt.savefig(plot_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
        plt.show()
        print(f"  → Elbow plot saved: {plot_path}")

    # ── Refit at optimal k ───────────────────────────────────────────────
    # While svd_full has the components, it's cleaner to have a model object strictly for k_opt
    # or we can just transform using the full model and slice, but `transform` on a new model is safer for pipeline consistency.
    svd_opt = TruncatedSVD(n_components=k_opt, random_state=42)
    Z = svd_opt.fit_transform(M)

    return Z, svd_opt, k_opt

In [ ]:
# Execution
Z, svd_model, k_opt = fit_svd_with_elbow(
    M,
    plot_path="/fig_svd_elbow.png"
)

print(f"\nFinal Latent Matrix Z shape: {Z.shape}")

## Define Semantic Labeling Utility

In [ ]:
def get_latent_semantics(svd_model, vocab):
    """
    Interpret SVD latent dimensions by identifying the top-weighted tags.

    Parameters
    ----------
    svd_model : TruncatedSVD object
    vocab     : list of tag strings corresponding to columns

    Returns
    -------
    dict : {dimension_index: "tag1, tag2, ..."}
    """
    semantics = {}
    for i, component in enumerate(svd_model.components_):
        # Get indices of top 5 positive weights
        top_indices = component.argsort()[::-1][:5]
        # Map to strings
        top_tags = [vocab[idx] for idx in top_indices]
        semantics[i] = ", ".join(top_tags)

    return semantics

# Execution
latent_labels = get_latent_semantics(svd_model, vocab)

print("Latent Semantics Component (Top 5):")
for i in range(k_opt):
    print(f"  Dimension {i}: {latent_labels[i]}")

# Visualization

## Mood Proportion Chart

In [ ]:
def plot_mood_profile(album_name: str, Z: np.ndarray, df: pd.DataFrame,
                      latent_labels: dict, top_k: int = 6):
    """
    Visualize the top semantic themes (moods) of a specific album based on its latent vector.
    """
    # Search for album
    mask = df["release_name"].str.contains(album_name, case=False, na=False)
    if not mask.any():
        print(f"Album '{album_name}' not found.")
        return
    idx = mask.idxmax()
    row = df.iloc[idx]
    album_title = row["release_name"]
    artist = row["artist_name"]

    # Get latent vector
    vec = Z[idx]

    # Identify top_k positive dimensions
    # We only care about positive activations
    top_indices = vec.argsort()[::-1][:top_k]
    top_scores = vec[top_indices]
    top_semantics = [latent_labels.get(i, f"Dim {i}") for i in top_indices]

    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(DARK)

    y_pos = np.arange(len(top_semantics))
    ax.barh(y_pos, top_scores, color=ACCENT[3], alpha=0.8, align='center', edgecolor="#1A1D23")

    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_semantics, fontsize=10, color="#E0E0E0")
    ax.invert_yaxis()

    ax.set_xlabel("Latent Semantic Score", fontsize=10)
    ax.set_title(f"Mood Profile: {album_title} — {artist}", fontsize=13, color="white", pad=12)

    plt.tight_layout()
    plt.show()

### Excution

In [ ]:
# Execution
plot_mood_profile("Abbey Road", Z, df, latent_labels)

## TSNE

In [ ]:
from sklearn.manifold import TSNE

def plot_latent_space(Z: np.ndarray, df: pd.DataFrame, highlight_albums=None,
                      save_path="fig_latent_tsne.png"):
    """
    t-SNE projection of the SVD latent space, colored by decade.
    """
    print(" Running t-SNE on latent vectors ...")
    tsne = TSNE(n_components=2, perplexity=40, random_state=42,
                learning_rate="auto", init="pca", max_iter=1000)
    emb = tsne.fit_transform(Z)

    # Create decade color mapping
    decades = (df["year"].fillna(2000) // 10 * 10).astype(int)
    unique_decades = sorted(decades.unique())
    cmap = {d: ACCENT[i % len(ACCENT)] for i, d in enumerate(unique_decades)}

    fig, ax = plt.subplots(figsize=(14, 10))
    fig.patch.set_facecolor(DARK)

    # Plot all points
    for d in unique_decades:
        mask = decades == d
        ax.scatter(emb[mask, 0], emb[mask, 1], c=cmap[d], s=15, alpha=0.4,
                   label=f"{d}s", edgecolors="none")

    # Highlight specific albums
    if highlight_albums:
        print(f"  Highlighting {len(highlight_albums)} albums...")
        for name in highlight_albums:
            # Find best match
            m = df["release_name"].str.contains(name, case=False, na=False)
            if m.any():
                idx = m.idxmax()
                ax.scatter(emb[idx, 0], emb[idx, 1], s=150, c="white",
                           edgecolors=ACCENT[0], linewidths=2.5, zorder=10)
                ax.annotate(df.iloc[idx]["release_name"], xy=emb[idx],
                            fontsize=9, color="white", fontweight="bold",
                            xytext=(8, 8), textcoords="offset points",
                            bbox=dict(boxstyle="round,pad=0.2", fc=ACCENT[0],
                                      alpha=0.8, edgecolor="none"))

    ax.set_title("Album Latent Space — t-SNE of SVD Embeddings",
                 fontsize=15, pad=12, color="white")
    ax.set_xticks([]); ax.set_yticks([])
    ax.grid(False)

    # Custom legend
    leg = ax.legend(fontsize=10, framealpha=0.2, ncol=1, loc="upper left",
                    bbox_to_anchor=(1, 1))
    for t in leg.get_texts():
        t.set_color("white")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"  → Saved t-SNE plot to: {save_path}")

### Excution

In [ ]:
# Execution

# t-SNE Plot
highlights = ["OK Computer"]
plot_latent_space(Z, df, highlight_albums=highlights,
                  save_path="fig_latent_tsne.png")

## Similarity Distribution

In [ ]:
def plot_similarity_distribution(query_name: str, Z: np.ndarray, df: pd.DataFrame,
                                 save_path=None):
    """
    Distribution of cosine similarities from the query album to all others.
    """
    mask = df["release_name"].str.contains(query_name, case=False, na=False)
    if not mask.any(): return
    q_idx = mask.idxmax()

    q_vec = Z[q_idx].reshape(1, -1)
    sims = cosine_similarity(q_vec, Z).flatten()
    sims[q_idx] = np.nan  # exclude self

    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor(DARK)

    ax.hist(sims[~np.isnan(sims)], bins=70, color=ACCENT[5], alpha=0.8,
            edgecolor="#1A1D23", linewidth=0.5)

    mean_sim = np.nanmean(sims)
    p95_sim = np.nanpercentile(sims, 95)

    ax.axvline(mean_sim, color=ACCENT[2], linestyle="--", linewidth=1.5,
               label=f"Mean: {mean_sim:.3f}")
    ax.axvline(p95_sim, color=ACCENT[0], linestyle="--", linewidth=1.5,
               label=f"95th %ile: {p95_sim:.3f}")

    ax.set_title(f"Similarity Distribution — {df.iloc[q_idx]['release_name']}",
                 fontsize=12, pad=10, color="white")
    ax.set_xlabel("Cosine Similarity", fontsize=10)
    ax.set_ylabel("Frequency", fontsize=10)

    leg = ax.legend(fontsize=9, framealpha=0.3)
    for t in leg.get_texts(): t.set_color("white")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    if save_path: print(f"  → Saved histogram to: {save_path}")

### Excution

In [ ]:
# Similarity Distributions
plot_similarity_distribution("OK Computer", Z, df, save_path="fig_simdist_ok.png")
plot_similarity_distribution("To Pimp a Butterfly", Z, df, save_path="fig_simdist_tpab.png")

## Artist Trajectory Map (PCA)

In [ ]:
from sklearn.decomposition import PCA

def plot_artist_trajectory(artist_name: str, Z: np.ndarray, df: pd.DataFrame, latent_labels: dict = None, save_path=None):
    """
    Visualize the stylistic evolution of an artist by projecting their albums'
    latent vectors into a 2D space (via PCA) and connecting them chronologically.
    """
    # 1. Filter and sort by NEW granular date object
    mask = df["artist_name"].str.contains(artist_name, case=False, na=False)
    # Refactored: Sort by 'date_obj' to ensure correct intra-year ordering
    artist_df = df[mask].sort_values("date_obj")

    if len(artist_df) < 3:
        print(f"  ⚠  Not enough albums found for '{artist_name}' (found {len(artist_df)}, need >= 3).")
        return

    # Extract latent vectors
    indices = artist_df.index.values
    local_Z = Z[indices]

    # Apply PCA for local 2D projection
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(local_Z)

    # Interpret the Axes
    if latent_labels:
        print(f"\n Analysing Trajectory Axes for {artist_df.iloc[0]['artist_name']}...")
        for i, comp_weights in enumerate(pca.components_):
            top_idxs = np.argsort(np.abs(comp_weights))[::-1][:5]
            print(f"  PC{i+1} (Axis {i+1}) is driven by:")
            for idx in top_idxs:
                w = comp_weights[idx]
                sign = "+" if w > 0 else "-"
                tags = latent_labels.get(idx, f"Dim {idx}")
                print(f"    • {sign}{abs(w):.2f} * [Dim {idx}: {tags}]")

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_facecolor(DARK)
    ax.scatter(coords[:, 0], coords[:, 1], c=ACCENT[5], s=100,
               edgecolor="white", zorder=5)

    # Connect with chronological arrows
    for i in range(len(coords) - 1):
        start = coords[i]
        end = coords[i+1]
        ax.annotate("", xy=end, xytext=start,
                    arrowprops=dict(arrowstyle="->", color=ACCENT[5],
                                    alpha=0.6, lw=1.5, connectionstyle="arc3,rad=0.1"))

    # Label points with Year-Month
    texts = []
    for i, row in enumerate(artist_df.itertuples()):
        date_str = row.date_obj.strftime('%Y-%m')
        label = f"{row.release_name}\n({date_str})"
        t = ax.text(coords[i, 0], coords[i, 1] + 0.02, label,
                    fontsize=9, color="white", ha="center", fontweight="bold")
        texts.append(t)

    # Adjust margins
    x_range = coords[:, 0].max() - coords[:, 0].min()
    y_range = coords[:, 1].max() - coords[:, 1].min()
    ax.set_xlim(coords[:, 0].min() - x_range*0.2, coords[:, 0].max() + x_range*0.2)
    ax.set_ylim(coords[:, 1].min() - y_range*0.2, coords[:, 1].max() + y_range*0.2)

    ax.set_title(f"Artist Trajectory: {artist_df.iloc[0]['artist_name']}",
                 fontsize=14, color="white", pad=15)
    ax.set_xlabel("PCA Component 1", fontsize=10)
    ax.set_ylabel("PCA Component 2", fontsize=10)
    ax.grid(True, alpha=0.1)

    plt.tight_layout()


    if save_path:
        plt.savefig(save_path)
    plt.show()

### Excution

In [ ]:
# Execution
plot_artist_trajectory("Pink Floyd", Z, df, latent_labels, save_path="artist_trajectory.png")

# Application

## Recommendation

In [ ]:
def recommend_by_index(query_idx: int, Z: np.ndarray, df: pd.DataFrame,
                       top_k: int = 15) -> pd.DataFrame:
    """
    Find the top_k most similar albums to the album at query_idx.
    """
    # Compute cosine similarity between query vector and all album vectors
    q_vec = Z[query_idx].reshape(1, -1)
    sims = cosine_similarity(q_vec, Z).flatten()

    # Exclude self from recommendations
    sims[query_idx] = -np.inf

    # Get top K indices
    top_indices = np.argsort(sims)[::-1][:top_k]

    # Format results
    results = []
    for rank, idx in enumerate(top_indices, 1):
        # Preview top 5 tags for context
        tags_preview = ", ".join(df.iloc[idx]["tags"][:5])
        results.append({
            "Rank":             rank,
            "Album":            df.iloc[idx]["release_name"],
            "Artist":           df.iloc[idx]["artist_name"],
            "Year":             df.iloc[idx].get("year", ""),
            "Rating":           df.iloc[idx].get("avg_rating", "N/A"),
            "Cosine_Sim":       round(sims[idx], 5),
            "Top_Descriptors":  tags_preview,
        })

    return pd.DataFrame(results).set_index("Rank")


def recommend_by_name(album_name: str, Z: np.ndarray, df: pd.DataFrame,
                      top_k: int = 15) -> pd.DataFrame:
    """
    Wrapper to search by album name (case-insensitive partial match).
    """
    mask = df["release_name"].str.contains(album_name, case=False, na=False)
    matches = df[mask]
    if matches.empty:
        print(f"  ⚠  No album matching '{album_name}'")
        return pd.DataFrame()

    # Take the first match
    idx = matches.index[0]
    row = df.iloc[idx]
    print(f"\n{row['release_name']} — {row['artist_name']}  ({row.get('year', '?')})")
    print(f"Tags: {', '.join(row['tags'][:10])} …")


    return recommend_by_index(idx, Z, df, top_k)


def recommend_by_keywords(query_tags: list, Z: np.ndarray, svd_model,
                          vocab: list, tag2col: dict, idf: np.ndarray,
                          df: pd.DataFrame, top_k: int = 15) -> pd.DataFrame:
    """
    Project a list of query tags into the latent space and find nearest albums.
    """
    n_vocab = len(vocab)
    q_sparse = np.zeros(n_vocab)

    # Clean tags and map to vocabulary
    tags_clean = [t.strip().lower().replace(" ", "") for t in query_tags]

    # Generate weights for query tags (assuming they are ordered by importance)
    n = len(tags_clean)
    weights = positional_decay_weights(n, alpha=0.10)

    matched_count = 0
    for pos, tag in enumerate(tags_clean):
        if tag in tag2col:
            col = tag2col[tag]
            q_sparse[col] = weights[pos] * idf[col]
            matched_count += 1

    if matched_count == 0:
        print("  ⚠  No query tags found in vocabulary.")
        return pd.DataFrame()

    # L2-normalize the sparse vector
    norm_val = norm(q_sparse)
    if norm_val > 0:
        q_norm = q_sparse / norm_val
    else:
        q_norm = q_sparse

    # Project into latent space: q_latent = q_sparse @ V^T
    # svd.components_ is V^T (shape: k x n_features)
    q_latent = q_norm @ svd_model.components_.T
    q_latent = q_latent.reshape(1, -1)

    # Compute cosine similarity
    sims = cosine_similarity(q_latent, Z).flatten()
    top_indices = np.argsort(sims)[::-1][:top_k]

    print(f"\n Keyword Query: {query_tags}  ({matched_count}/{len(tags_clean)} matched)")

    results = []
    for rank, idx in enumerate(top_indices, 1):
        results.append({
            "Rank":            rank,
            "Album":           df.iloc[idx]["release_name"],
            "Artist":          df.iloc[idx]["artist_name"],
            "Year":            df.iloc[idx].get("year", ""),
            "Rating":          df.iloc[idx].get("avg_rating", "N/A"),
            "Cosine_Sim":      round(sims[idx], 5),
            "Top_Descriptors": ", ".join(df.iloc[idx]["tags"][:5]),
        })

    return pd.DataFrame(results).set_index("Rank")

###Excution

In [ ]:
# Execution

# Album Recommendation
print("Album-to-Album Recommendation Test")
recs = recommend_by_name("OK Computer", Z, df, top_k=10)
if not recs.empty:
    display(recs)


# Keyword Recommendation
print("\n Keyword-to-Album Recommendation Test")
query_keywords = ["dark", "atmospheric", "cold", "melancholic"]
recs_keywords = recommend_by_keywords(
    query_keywords, Z, svd_model, vocab, tag2col, idf, df, top_k=10
)
if not recs_keywords.empty:
    display(recs_keywords)